In [2]:
include("../src/TensorDecomposition.jl")
using LinearAlgebra, LinearSolve

## Example for refinement of Jennrich's

In [3]:
n = 5
r = 12
d = 5

D, Drev = TensorDecomposition.makeDicts(n, d);
basis_inds = collect(1:r)
basis, basisD = TensorDecomposition.basisFn(basis_inds, Drev);

In [4]:
Z = randn(n+1, r)
T = TensorDecomposition.rankedTensor(ones(r), Z, d; type=eltype(Z));
Tzero = TensorDecomposition.catMat(T, d);

Tcat = TensorDecomposition.catMat(T, 2)
H0 = Tcat[basis_inds, basis_inds]

Hs = []
for i=1:n
    basis_i = [TensorDecomposition.multMon(b, i) for b in basis]
    Hi = Array{eltype(T), 2}(undef, length(basis), length(basis_i))
    for (j, alpha1) in enumerate(basis)
        for (k, alpha2) in enumerate(basis_i)
            gamma = alpha1+alpha2 
            Hi[j, k] = Tzero[D[gamma], 1]
        end 
    end
    push!(Hs, Hi)
end

Ns = []
for H in Hs 
    push!(Ns, H*inv(H0))
end

$n=6, r=12$, so for Jennrich's to work we need $r\leq \binom{5+2}{5}=21$.

In [5]:
Z

6×12 Matrix{Float64}:
 -0.478465    0.711888    0.252508  …  -0.164986    0.186277   1.54763
  0.655858   -0.154264    2.09191      -0.314402   -0.580148   0.386863
 -0.930424    0.0641174   2.15146      -0.602619   -0.73178   -0.935537
 -0.0110791   0.660673    1.29319       0.0846993   0.102952   0.457215
  0.807722    1.03712    -0.597252     -0.989526    0.272552  -0.689806
 -0.85144    -0.966542   -0.872327  …   0.604926   -1.81841   -0.710127

In [6]:
T

6×6×6×6×6 Array{Float64, 5}:
[:, :, 1, 1, 1] =
 -11.3808    4.84373   -17.3332   -6.76814   -1.39764   -23.94
   4.84373  -3.7728      4.42763  -1.66953   -0.464909    4.32497
 -17.3332    4.42763    -6.24261  -1.10915    1.36444    -5.65653
  -6.76814  -1.66953    -1.10915  -5.78136    0.851294   -0.108612
  -1.39764  -0.464909    1.36444   0.851294   1.22979    -2.30592
 -23.94      4.32497    -5.65653  -0.108612  -2.30592     1.654

[:, :, 2, 1, 1] =
  4.84373   -3.7728     4.42763    -1.66953   -0.464909    4.32497
 -3.7728    -1.34817    0.401967   -2.37489   -0.192693   -1.29801
  4.42763    0.401967   3.15749     2.71318   -0.0905172   2.86817
 -1.66953   -2.37489    2.71318    -3.17273    0.147783    0.989033
 -0.464909  -0.192693  -0.0905172   0.147783   0.41597     0.623079
  4.32497   -1.29801    2.86817     0.989033   0.623079    3.16294

[:, :, 3, 1, 1] =
 -17.3332    4.42763    -6.24261   -1.10915    1.36444     -5.65653
   4.42763   0.401967    3.15749    2.71318   -0.09

We take $B$ to be the first $r$ monomials, so that
$$B= \{1, x_1, x_2, x_3, x_4, x_5, x_1^2, x_1x_2, x_1x_3, x_1x_4, x_1x_5, x_2^2\}. $$
The corresponding $\mathbf{H}_B$ is invertble.

In [7]:
basis

12-element Vector{Vector{Int64}}:
 [0, 0, 0, 0, 0]
 [1, 0, 0, 0, 0]
 [0, 1, 0, 0, 0]
 [0, 0, 1, 0, 0]
 [0, 0, 0, 1, 0]
 [0, 0, 0, 0, 1]
 [2, 0, 0, 0, 0]
 [1, 1, 0, 0, 0]
 [1, 0, 1, 0, 0]
 [1, 0, 0, 1, 0]
 [1, 0, 0, 0, 1]
 [0, 2, 0, 0, 0]

In [8]:
svdvals(H0)

12-element Vector{Float64}:
 47.1787430388109
 23.27824138144627
 17.21299356700469
 12.300613872999325
  4.444527487542688
  3.1420760642436796
  2.5939009584609285
  0.4260995285250003
  0.27923799337882177
  0.21861194454752067
  0.10954664708354483
  0.01733497427280146

We then form the matrices $\mathbf{H}_{B_i}$.  As an example, the following is $\mathbf{H}_{B_1}$.

In [9]:
Hs[1]

12×12 Matrix{Float64}:
  4.84373   -3.7728     4.42763    -1.66953   …   -1.29801      3.15749
 -3.7728    -1.34817    0.401967   -2.37489       -0.816912     3.63986
  4.42763    0.401967   3.15749     2.71318       -3.15857      7.32503
 -1.66953   -2.37489    2.71318    -3.17273       -1.1351       2.89575
 -0.464909  -0.192693  -0.0905172   0.147783       0.791556    -1.91656
  4.32497   -1.29801    2.86817     0.989033  …   -0.0729904   -0.0139498
 -1.34817    0.789757   6.64656     1.22863      -16.2659      42.1386
  0.401967   6.64656    3.63986     4.18956      -17.2393      42.8557
 -2.37489    1.22863    4.18956    -2.01434      -10.7234      25.2626
 -0.192693  -2.44254   -1.32377    -0.104852       4.80987    -11.7097
 -1.29801   -0.816912  -3.15857    -1.1351    …    6.49945    -18.79
  3.15749    3.63986    7.32503     2.89575      -18.79        46.2694

Then the decomposition can be recovered from $\mathbf{N}_i$.

In [10]:
N = sum([randn()*N_ for N_ in Ns])
Zhat_ = eigvecs(N)[1:n+1, :]
Zhat = Zhat_ ./ permutedims(Zhat_[1, :])
lhat = TensorDecomposition.khatri_rao(Zhat, d; type=eltype(Zhat)) \ reshape(T, (n+1)^d);

maximum(abs.(T-TensorDecomposition.rankedTensor(lhat, Zhat, d, type=eltype(Zhat))))/maximum(abs.(T))

1.4176052773106793e-13

## Example for efficient moment matrix extension

In [11]:
n = 2
r = 4

D, Drev = TensorDecomposition.makeDicts(n, 4);
basis_inds = collect(1:r)
basis, basisD = TensorDecomposition.basisFn(basis_inds, Drev);

vars = TensorDecomposition.varTups(basis, n, 4)
eqs1, eqs2 = TensorDecomposition.linEqTups(basisD, n, 4);

In [15]:
Z = rand(-5:5, n+1, r)
T = TensorDecomposition.rankedTensor(ones(r), Z, 4; type=eltype(Z));

Tcat = TensorDecomposition.catMat(T, 2)
H0 = Tcat[basis_inds, basis_inds]

Z

3×4 Matrix{Int64}:
 -1   2  -4  -5
 -2   0  -5   0
  3  -4   3  -2

In [24]:
D

Dict{Any, Any} with 15 entries:
  [0, 0] => 1
  [1, 1] => 5
  [1, 0] => 2
  [0, 3] => 10
  [2, 1] => 8
  [1, 3] => 14
  [2, 0] => 4
  [3, 1] => 12
  [3, 0] => 7
  [4, 0] => 11
  [0, 4] => 15
  [0, 2] => 6
  [1, 2] => 9
  [2, 2] => 13
  [0, 1] => 3

The following are $\mathbf{H}, \mathbf{H}_{B_1}, \mathbf{H}_{B_2}$, respectively.

<style>
td, th {
   border: none!important;
}
</style>

|         | $1$ | $x_1$ | $x_2$ | $x_1^2$ |
|---------|-----|-------|-------|---------|
| $1$     | 898 | 322   | 23    | 404     |
| $x_1$   | 322 | 404   | -246  | 508     |
| $x_2$   | 23  | -246  | 317   | -312    |
| $x_1^2$ | 404 | 508   | -312  | 641     |

<br>

|         | $x_1$ | $x_1^2$ | $x_1x_2$ | $x_1^3$ |
|---------|-----|-------|-------|---------|
| $1$     | 322 | 404   | -246    | 508     |
| $x_1$   | 404 | 508   | -312  | 641     |
| $x_2$   | -246 | -312  | 198 | -399 |
| $x_1^2$ | 508 | 641   | -189  | $y_{x_1^5}$ |

<br>

|         | $x_2$ | $x_1x_2$ | $x_2^2$ | $x_1^2x_2$ |
|---------|-----|-------|-------|---------|
| $1$     | 23 | -246   | 317    | -312     |
| $x_1$   | -246 | -312   | 198  | -189     |
| $x_2$   | 317 | 198  | -223 | 261 |
| $x_1^2$ | -312 | -189   | 261  | $y_{x_1^4x_2}$ |

In [31]:
A, b = TensorDecomposition.linearSystem(T, H0, basis_inds, basisD, D, vars, eqs1, eqs2; type=eltype(T));

In [32]:
A

2×2 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4 stored entries:
  3.0   1.0
 -3.8  -3.0

In [33]:
b

2-element Vector{Float64}:
  1922.999999999981
 -1540.099999999984

In [34]:
A_ = Matrix(copy(A))
foreach(TensorDecomposition.normalize!, eachcol(A_));
foreach(TensorDecomposition.normalize!, eachrow(A_));
svdvals(A_)

2-element Vector{Float64}:
 1.38492823399529
 0.2863106471696209

In [35]:
prob = LinearProblem(A, b)
sol = solve(prob)
solDict = Dict([v => s for (v, s) in zip(vars, sol.u)]);
Ms = TensorDecomposition.multMatrices(T, basis, solDict, D, H0)
lhat, Zhat = TensorDecomposition.obtainDecomp(T, Ms);

In [36]:
maximum(abs.(T-TensorDecomposition.rankedTensor(lhat, Zhat, 4, type=eltype(Zhat))))/maximum(abs.(T))

2.7219009031344594e-14